> **Historical notebook: seven-temperature Wolff references (L=32).**
> This notebook provides reference statistics and visualisations from the earlier Wolff / supervised Flow Matching stage; it does not train or evaluate the current data-free SPS model across temperatures. The final section adds the same historical FM diagnostic used in Notebook 02.
> The original scientific code is retained, with saved outputs removed for publication. A clean clone does not contain the seven `data/xy_L32_T*_N1000.pt` tensors (1000 configurations per temperature), the historical checkpoint, or the raw comparison arrays. Generate new reference tensors with `src/generate_wolff_sweep.py`, or supply the original experiment files.
> Read the [historical baseline guide](../docs/public/legacy.md) for selected figures and interpretation limits.


# 03 — Multi-Temperature Quantitative BKT Observables

This notebook implements Zhou's request: do **not** rely on a single image or on
the trivial `Q_net = 0` identity.  Instead, compute ensemble-averaged
observables for a temperature sweep and plot them with error bars.

Temperatures: `0.5, 0.7, 0.8, 0.89, 1.0, 1.2, 1.5`  
Samples per T: `1000` Wolff configurations (L=32).


说明：误差棒是构型间的标准差 SD，不是均值置信区间。能量是每键能量；“涡旋对数”是正负涡旋总数除以二，不代表已经识别束缚对。50 次聚类更新的间隔尚未经过自相关验证，因此图中的样本不应直接宣称独立。


In [ ]:
from pathlib import Path

# Find the project from either the repository root or notebooks/.
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'src' / 'model_dit.py').is_file() and (p / 'data').is_dir()
)
FIGURE_DIR = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

import os
import numpy as np
import torch
import matplotlib.pyplot as plt

%matplotlib inline

L = 32
T_LIST = [0.5, 0.7, 0.8, 0.89, 1.0, 1.2, 1.5]
DATA_DIR = PROJECT_ROOT / 'data'


def data_path_for_T(T):
    tag = f'{T:g}'.replace('.', '_')
    return os.path.join(DATA_DIR, f'xy_L32_T{tag}_N1000.pt')


# Shared physical definitions: same units and vortex convention as notebook 02.
import sys
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from xy_observables import calculate_vorticity, vortex_pairs, energy_density


In [ ]:
records = []
for T in T_LIST:
    tensor = torch.load(data_path_for_T(T), weights_only=False)
    theta = torch.atan2(tensor[:, 1], tensor[:, 0]).numpy()

    pairs = vortex_pairs(theta)
    energies = energy_density(theta)

    records.append({
        'T': T,
        'pairs_mean': float(pairs.mean()),
        'pairs_std': float(pairs.std(ddof=1)),
        'energy_mean': float(energies.mean()),
        'energy_std': float(energies.std(ddof=1)),
        'theta': theta,
    })
    print(f'T={T:<5} | pairs = {pairs.mean():6.2f} +/- {pairs.std(ddof=1):5.2f} '
          f'| energy = {energies.mean():7.4f} +/- {energies.std(ddof=1):6.4f}')


In [ ]:
Ts = np.array([r['T'] for r in records])
pairs_mean = np.array([r['pairs_mean'] for r in records])
pairs_std = np.array([r['pairs_std'] for r in records])
energy_mean = np.array([r['energy_mean'] for r in records])
energy_std = np.array([r['energy_std'] for r in records])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].errorbar(Ts, pairs_mean, yerr=pairs_std, marker='o', ms=6,
                 capsize=4, color='#1f77b4', lw=2, label='Wolff ensemble (mean +/- SD)')
axes[0].axvline(0.89, color='red', ls='--', lw=1.5, label='T=0.89 (BKT region)')
axes[0].set_xlabel('Temperature T')
axes[0].set_ylabel('Mean vortex pairs')
axes[0].set_title('Vortex count vs T (L=32)')
axes[0].grid(True, linestyle=':', alpha=0.7)
axes[0].legend()

axes[1].errorbar(Ts, energy_mean, yerr=energy_std, marker='s', ms=6,
                 capsize=4, color='#ff7f0e', lw=2, label='Wolff ensemble (mean +/- SD)')
axes[1].axvline(0.89, color='red', ls='--', lw=1.5, label='T=0.89 (BKT region)')
axes[1].set_xlabel('Temperature T')
axes[1].set_ylabel('Energy per bond')
axes[1].set_title('Energy density vs T (L=32)')
axes[1].grid(True, linestyle=':', alpha=0.7)
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'multitemp_vortex_energy.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURE_DIR / 'multitemp_vortex_energy.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Show three distinct saved examples per T, so the visualisation does not rely on
# a single configuration.
rng = np.random.default_rng(42)
n_cols = 3
fig, axes = plt.subplots(len(records), n_cols, figsize=(4.2 * n_cols, 3.4 * len(records)))

for row, rec in enumerate(records):
    T = rec['T']
    theta = rec['theta']
    idx = rng.choice(theta.shape[0], size=n_cols, replace=False)

    for col in range(n_cols):
        ax = axes[row, col]
        th = theta[idx[col]]
        q = calculate_vorticity(th)
        n_pos = int((q == 1).sum())
        n_neg = int((q == -1).sum())

        ax.imshow(th, cmap='hsv', origin='lower', vmin=-np.pi, vmax=np.pi,
                  interpolation='nearest')
        pos = np.argwhere(q == 1)
        neg = np.argwhere(q == -1)
        if len(pos):
            ax.plot(pos[:, 1] + 0.5, pos[:, 0] + 0.5, 'o', color='red',
                    ms=5, markeredgecolor='darkred', markeredgewidth=0.8)
        if len(neg):
            ax.plot(neg[:, 1] + 0.5, neg[:, 0] + 0.5, 'o', color='blue',
                    ms=5, markeredgecolor='darkblue', markeredgewidth=0.8)

        ax.set_title(f'T={T} | +1:{n_pos} -1:{n_neg}', fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle('Wolff Configurations Across the BKT Region (L=32)',
             fontsize=14, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(FIGURE_DIR / 'multitemp_example_grid.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURE_DIR / 'multitemp_example_grid.pdf', dpi=300, bbox_inches='tight')
plt.show()


## 与 notebook 02 接起来：模型相对参考差在哪里

上面只有 Wolff，是参考背景。下面读取与 notebook 02 完全相同的保存实验，叠加当前 FM 的结果；不是重新训练模型，也不是为每个温度训练了模型。

主对照使用原训练数据源码的 β=1.12（T=0.892857…），独立新生成 4 条 Wolff 链共 1024 个样本。严格 T=0.89 的旧扫描点保留为背景。首次运行本节将准备数据包，以后直接读取并验证来源。

上排阴影是扫描构型的标准差 SD；FM 与新 Wolff 点上的误差棒是均值的 95% 抽样区间。下排展示同温度下的整批数量与能量分布；样本比例已经归一化。Wolff 的区间采用链内分块，并附带有限的相关性检查。详见 [historical baseline guide](../docs/public/legacy.md)。


In [ ]:
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from compare_fm_wolff import ensure_run, plot_comparison, print_summary, DEFAULT_OUTPUT

report, saved = ensure_run()
print_summary(report)
comparison = plot_comparison(report, saved)
for extension in ('png', 'pdf'):
    comparison.savefig(DEFAULT_OUTPUT / f'comparison.{extension}', dpi=180)
plt.show()
